In [17]:
import pypsa
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import geopandas as gpd
from pypsa.plot import add_legend_lines, add_legend_patches, add_legend_semicircles
import yaml
from pathlib import Path
import pandas as pd
import yaml


**Set Up**

In [18]:
fn = 'resources/DK_test/networks/base_s_2__12h_2050.nc'


In [19]:
n= pypsa.Network(fn)

config = yaml.safe_load(Path("config/config.denmark.yaml").read_text())


INFO:pypsa.network.io:New version 1.0.7 available! (Current: 0.35.2)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, links, loads, stores


In [20]:
p = Path(fn)  
try:
    if p.exists():
        p.unlink()
        print(f"Deleted {p}")
    else:
        print(f"File not found: {p}")
except Exception as e:
    print(f"Failed to delete {p}: {e}")

Deleted resources/DK_test/networks/base_s_2__12h_2050.nc


**Options**

In [21]:
ongrid=False
cluster_cost_reduction=1
cluster_size=800   #MW, it's the maximum installable capacity for each renewable in renewables in each country cluster
renewables={"solar-hsat","solar"}

In [ ]:
nodes_with_clusters = n.buses.loc[
    n.buses.index.str[:2].isin(config['countries']) &
    (n.buses['carrier'] == 'AC')
].index.tolist()




['DK0 0', 'DK1 0']

**Buses and Generators of the Cluster Addition**

In [36]:


def assign_cluster_generators_and_electricity_buses(n, config, cluster_size, cluster_cost_reduction, renewables):
    
    countries_renewables_cf = {}                #dictionary of dataframes by country and renewable type, sorting the generators by average capacity factor (ascending order)
    clusters_generators={}                      #dictionary of dataframes by country and renewable type, containing the generators assigned to the cluster  

    for country in config['countries']:
        for renewable in renewables:

            countries_renewables_cf[(country, renewable)] = pd.DataFrame(
                index=n.generators['p_nom_max'].loc[n.generators.index.astype(str).str.contains(rf"{country}.*{renewable}$")].index,
                columns=["p_max_pu","p_nom_max"]  
            )

            clusters_generators[(country, renewable)] = pd.DataFrame()

            #we are considering the highest mean p_min_pu to determine the best generators per renewable available

            countries_renewables_cf[(country, renewable)] ["p_max_pu"] = n.generators_t['p_max_pu'].loc[:, n.generators_t['p_max_pu'].columns.astype(str).str.contains(rf"{country}.*{renewable}$")].mean()
            countries_renewables_cf[(country, renewable)] ["p_nom_max"] = n.generators['p_nom_max'].loc[n.generators.index.astype(str).str.contains(rf"{country}.*{renewable}$")]

            countries_renewables_cf[(country, renewable)] = countries_renewables_cf[(country, renewable)].sort_values("p_max_pu", ascending=False)

            #print(countries_renewables_cf[(country, renewable)])

            number_gen=0



            while  countries_renewables_cf[(country, renewable)].iloc[0:number_gen+1]["p_nom_max"].sum() <= cluster_size:

                if number_gen >= len(countries_renewables_cf[(country, renewable)]):
                    raise ValueError(f"Not enough {renewable} generators to reach cluster_size.")
                
                number_gen=number_gen+1

            #print(f"{renewable} generators in cluster: {number_gen+1}")

            clusters_generators[(country, renewable)]  = n.generators.loc[countries_renewables_cf[(country, renewable)].index[0:number_gen+1]]
            remaining_capacity = countries_renewables_cf[(country, renewable)].iloc[0:number_gen+1]["p_nom_max"].sum() - cluster_size
            #countries_renewables_cf[(country, renewable)].iloc[number_gen]["p_nom_max"] = remaining_capacity maybe it is better to do this step later

            print(f"Remaining top {renewable} capacity outside the cluster: {remaining_capacity} MW")

            
            clusters_generators[(country, renewable)].loc[clusters_generators[(country, renewable)].index[number_gen], "p_nom_max"] = cluster_size - clusters_generators[(country, renewable)].loc[clusters_generators[(country, renewable)].index[0:number_gen],"p_nom_max"].sum()

            print(f"Capacity of the last {renewable} generator adjusted to fit cluster size: {clusters_generators[(country, renewable)].loc[clusters_generators[(country, renewable)].index[number_gen], 'p_nom_max']} MW")

            print(clusters_generators[(country, renewable)])

            for idx in clusters_generators[(country, renewable)].index:

                ### Electricity bus and generators ###

                if not n.buses.index.str.contains(rf"{clusters_generators[(country, renewable)].loc[idx].bus + " cluster"}$").any():
        
                    n.add(
                        "Bus",
                        name=clusters_generators[(country, renewable)].loc[idx].bus + " cluster",
                        v_nom=n.buses.at[clusters_generators[(country, renewable)].loc[idx].bus, "v_nom"],
                        x=n.buses.at[clusters_generators[(country, renewable)].loc[idx].bus, "x"],
                        y=n.buses.at[clusters_generators[(country, renewable)].loc[idx].bus, "y"],
                        unit=n.buses.at[clusters_generators[(country, renewable)].loc[idx].bus, "unit"],
                        location=n.buses.at[clusters_generators[(country, renewable)].loc[idx].bus, "location"],
                        country=n.buses.at[clusters_generators[(country, renewable)].loc[idx].bus, "country"],
                        carrier=n.buses.at[clusters_generators[(country, renewable)].loc[idx].bus, "carrier"],
                        control=n.buses.at[clusters_generators[(country, renewable)].loc[idx].bus, "control"],
                        substation_lv=n.buses.at[clusters_generators[(country, renewable)].loc[idx].bus, "substation_lv"],
                        substation_off=n.buses.at[clusters_generators[(country, renewable)].loc[idx].bus, "substation_off"],
                    )

                n.add(
                    "Generator",
                    name=clusters_generators[(country, renewable)].loc[idx].name + " cluster",
                    bus=clusters_generators[(country, renewable)].loc[idx].bus + " cluster",
                    carrier=clusters_generators[(country, renewable)].loc[idx].carrier,
                    p_nom_max=clusters_generators[(country, renewable)].loc[idx].p_nom_max,
                    p_max_pu=clusters_generators[(country, renewable)].loc[idx].p_max_pu,
                    marginal_cost=clusters_generators[(country, renewable)].loc[idx].marginal_cost*(1-cluster_cost_reduction),
                    capital_cost=clusters_generators[(country, renewable)].loc[idx].capital_cost*(1-cluster_cost_reduction),
                    efficiency=clusters_generators[(country, renewable)].loc[idx].efficiency,
                    p_nom_extendable=True,
                    overwrite=True,)
                
                n.generators_t['p_max_pu'][clusters_generators[(country, renewable)].loc[idx].name + " cluster"] = n.generators_t['p_max_pu'][clusters_generators[(country, renewable)].loc[idx].name]


                ### H2 bus ##

                if not n.buses.index.str.contains(rf"{clusters_generators[(country, renewable)].loc[idx].bus + " H2 cluster"}$").any():

                    n.add(
                        "Bus",
                        name=clusters_generators[(country, renewable)].loc[idx].bus + " H2 cluster",
                        v_nom=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " H2"}", "v_nom"],
                        x=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " H2"}", "x"],
                        y=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " H2"}", "y"],
                        unit=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " H2"}", "unit"],
                        location=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " H2"}", "location"],
                        country=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " H2"}", "country"],
                        carrier=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " H2"}", "carrier"],
                        control=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " H2"}", "control"],
                        substation_lv=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " H2"}", "substation_lv"],
                        substation_off=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " H2"}", "substation_off"],
                    )
                
                ### Batteries bus ###

                if not n.buses.index.str.contains(rf"{clusters_generators[(country, renewable)].loc[idx].bus + " battery cluster"}$").any():

                    n.add(
                        "Bus",
                        name=clusters_generators[(country, renewable)].loc[idx].bus + " battery cluster",
                        v_nom=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " battery"}", "v_nom"],
                        x=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " battery"}", "x"],
                        y=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " battery"}", "y"],
                        unit=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " battery"}", "unit"],
                        location=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " battery"}", "location"],
                        country=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " battery"}", "country"],
                        carrier=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " battery"}", "carrier"],
                        control=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " battery"}", "control"],
                        substation_lv=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " battery"}", "substation_lv"],
                        substation_off=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " battery"}", "substation_off"],
                    )



                if idx == countries_renewables_cf[(country, renewable)].iloc[number_gen].name:
                    n.generators.loc[n.generators.index == idx, "p_nom_max"] = countries_renewables_cf[(country, renewable)].iloc[0:number_gen+1]["p_nom_max"].sum() - cluster_size

                    print(f"Residual capacity of generator {clusters_generators[(country, renewable)].loc[idx].name} is {n.generators.loc[n.generators.index == idx, 'p_nom_max']} MW")
                
                else:


                    n.remove(
                            "Generator",
                            name=clusters_generators[(country, renewable)].loc[idx].name,
                    )

    nodes_with_clusters = (n.buses.loc[n.buses.index.str.contains("cluster"), [ "country", "location"]].drop_duplicates().reset_index(drop=True)
)



            

    return n, nodes_with_clusters

n, nodes_with_clusters = assign_cluster_generators_and_electricity_buses(n, config, cluster_size, cluster_cost_reduction, renewables)
            



            

            





        




Remaining top solar-hsat capacity outside the cluster: 242577.01960275497 MW
Capacity of the last solar-hsat generator adjusted to fit cluster size: 800.0 MW
                      bus control type  p_nom  p_nom_mod  p_nom_extendable  \
Generator                                                                    
IT0 1 0 solar-hsat  IT0 1      PQ         0.0        0.0              True   

                    p_nom_min  p_nom_max  p_min_pu  p_max_pu  ...  \
Generator                                                     ...   
IT0 1 0 solar-hsat        0.0      800.0       0.0       1.0  ...   

                    up_time_before  down_time_before  ramp_limit_up  \
Generator                                                             
IT0 1 0 solar-hsat               1                 0            NaN   

                    ramp_limit_down  ramp_limit_start_up ramp_limit_shut_down  \
Generator                                                                       
IT0 1 0 solar-hsat     

In [37]:
n.buses

,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network,country,substation_lv,substation_off
Bus,,,,,,,,,,,,,,,,
IT0 0,380.0,,9.417137,45.416172,AC,MWh_el,IT0 0,1.0,0.0,inf,Slack,,,IT,1.0,1.0
IT0 1,380.0,,15.089899,40.035720,AC,MWh_el,IT0 1,1.0,0.0,inf,PQ,,,IT,1.0,1.0
IT0 2,380.0,,12.311532,44.283118,AC,MWh_el,IT0 2,1.0,0.0,inf,PQ,,,IT,1.0,1.0
IT1 0,380.0,,8.739825,40.032890,AC,MWh_el,IT1 0,1.0,0.0,inf,Slack,,,IT,1.0,1.0
EU,1.0,,-5.500000,46.000000,none,,EU,1.0,0.0,inf,PQ,,,,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
IT1 0 urban decentral heat,1.0,,8.739825,40.032890,urban decentral heat,MWh_th,IT1 0,1.0,0.0,inf,PQ,,,IT,NaN,NaN
IT1 0 urban decentral water tanks,1.0,,8.739825,40.032890,urban decentral water tanks,MWh_th,IT1 0,1.0,0.0,inf,PQ,,,IT,NaN,NaN
IT0 1 cluster,380.0,,15.089899,40.035720,AC,MWh_el,IT0 1,1.0,0.0,inf,PQ,,,IT,1.0,1.0


**Links of the Cluster Addition**

In [38]:
n.links

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
relation/8185711-500-DC,IT1 0,IT0 2,,DC,0.967388,True,0,inf,1000.0,0.0,...,0.0,relation/8185711,LINESTRING (8.306507186431089 40.8411086571579...,1.0,0.961843,,NaN,,False,556.670095
TYNDP2024_1157,IT0 0,IT0 2,,DC,0.974075,True,2030,inf,0.0,0.0,...,1.0,"{name:HG North Tyrrhenian Corridor,url:{name:T...","LINESTRING (9.409 45.553, 12.015 42.244)",NaN,0.000000,in_permitting,NaN,,False,260.622634
TYNDP2024_1166,IT0 2,IT0 1,,DC,0.968106,True,2036,inf,0.0,0.0,...,1.0,"{name:HG Adriatic Corridor,url:{name:TYNDP2024...","LINESTRING (11.661 44.855, 15.55 41.513)",NaN,0.330000,in_permitting,NaN,,False,524.800355
TYNDP2024_1168,IT0 1,IT0 2,,DC,0.968106,True,2035,inf,0.0,0.0,...,1.0,"{name:HG Ionian-Tyrrhenian Corridor,url:{name:...","LINESTRING (16.629 39.568, 12.779 41.43)",NaN,0.520000,in_permitting,NaN,,False,524.800355
IT0 0 co2 sequestered,IT0 0 co2 stored,IT0 0 co2 sequestered,,co2 sequestered,1.000000,True,0,inf,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
IT1 0 urban decentral biomass boiler,IT1 0 solid biomass,IT1 0 urban decentral heat,,urban decentral biomass boiler,0.860000,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.000000
IT1 0 urban decentral gas boiler,IT1 0 gas,IT1 0 urban decentral heat,,urban decentral gas boiler,0.980000,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.000000
IT1 0 urban decentral resistive heater,IT1 0 low voltage,IT1 0 urban decentral heat,,urban decentral resistive heater,0.900000,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.000000


In [39]:
def add_cluster_links(n, nodes_with_clusters, cluster_cost_reduction, ongrid):

    for node in nodes_with_clusters["location"]:

        ### H2 Electrolysis ###

        link_name = f"{node} H2 Electrolysis"

        n.add(
            "Link",
            name=link_name + " cluster",
            bus0=n.links.at[link_name, "bus0"] + " cluster",
            bus1=n.links.at[link_name, "bus1"] + " cluster",
            p_nom_extendable=n.links.at[link_name, "p_nom_extendable"],
            carrier=n.links.at[link_name, "carrier"],
            efficiency=n.links.at[link_name, "efficiency"],
            capital_cost=n.links.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.links.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            lifetime=n.links.at[link_name, "lifetime"],
            reversed=False,
            overwrite=True,
        )

        ### Methanolization ###

        link_name = f"{node} methanolisation"

        n.add(
            "Link",
            name=link_name + " cluster",
            bus0=n.links.at[link_name, "bus0"] + " cluster",
            bus1=n.links.at[link_name, "bus1"],
            bus2=n.links.at[link_name, "bus2"] + " cluster",
            bus3=n.links.at[link_name, "bus3"],
            bus4=n.links.at[link_name, "bus4"],
            p_nom_extendable=n.links.at[link_name, "p_nom_extendable"],
            p_min_pu=n.links.at[link_name, "p_min_pu"],
            carrier=n.links.at[link_name, "carrier"],
            efficiency=n.links.at[link_name, "efficiency"],
            efficiency2=n.links.at[link_name, "efficiency2"],
            efficiency3=n.links.at[link_name, "efficiency3"],
            efficiency4=n.links.at[link_name, "efficiency4"],
            capital_cost=n.links.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.links.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            lifetime=n.links.at[link_name, "lifetime"],
            reversed=False,
            overwrite=True,
        )


    if ongrid==True :

        ### Electricity connection to grid ###

        link_name = f"{node} electricity cluster"
        
        n.add(
            "Link",
            name=link_name,
            bus0=f"{node} cluster",
            bus1=f"{node}",
            carrier=n.buses.at[f"{node}", "carrier"],  
            p_nom_extendable=True,
            efficiency=1.0,
            capital_cost=0.0,
            marginal_cost=0.0,
            reversed=False,
            overwrite=True,
        )

        link_name = f"{node} electricity cluster back"
        n.add(
            "Link",
            name=link_name,
            bus0=f"{node}",
            bus1=f"{node} cluster",
            carrier=n.buses.at[f"{node}", "carrier"],  
            p_nom_extendable=True,
            efficiency=1.0,
            capital_cost=0.0,
            marginal_cost=0.0,
            reversed=True,
            overwrite=True,
        )

    else:
        if f"{node} cluster electricity" in n.links.index:
            n.remove(
                "Link",
                name=f"{node} cluster electricity",
            )
        if f"{node} cluster electricity back" in n.links.index:
            n.remove(
                "Link",
                name=f"{node} cluster electricity back",
            )

    return n

n = add_cluster_links(n, nodes_with_clusters, cluster_cost_reduction, ongrid)



        


        

**Storages of the Cluster Addition**

In [40]:
def add_cluster_storages(n, nodes_with_clusters, cluster_cost_reduction):

    for node in nodes_with_clusters["location"]:

        link_name = f"{node} H2 Store"

    
        n.add("Store",
            name=link_name + " cluster",
            bus=n.stores.at[link_name, "bus"] + " cluster",
            carrier=n.stores.at[link_name, "carrier"],
            e_nom_extendable=True,
            capital_cost=n.stores.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.stores.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            e_initial_per_period=n.stores.at[link_name, "e_initial_per_period"],
            e_cyclic=n.stores.at[link_name, "e_cyclic"],
            e_cyclic_per_period=n.stores.at[link_name, "e_cyclic_per_period"],
            overwrite=True,
            )
        
        link_name = f"{node} battery"


        n.add(
                "Link",
                name=link_name + " charger cluster",
                bus0=f"{node} cluster",
                bus1=f"{node} battery cluster",
                carrier=n.buses.at[link_name, "carrier"],   
                p_nom_extendable=True,
                efficiency=1.0,
                capital_cost=0.0,
                marginal_cost=0.0,
                reversed=False,
                overwrite=True,
            )
        n.add(
                "Link",
                name=link_name + " discharger cluster",
                bus0=f"{node} battery cluster",
                bus1=f"{node} cluster",
                carrier=n.buses.at[link_name, "carrier"],
                p_nom_extendable=True,
                efficiency=1.0,
                capital_cost=0.0,
                marginal_cost=0.0,
                reversed=True,
                overwrite=True,
            )

        n.add("Store",
            name=link_name + " cluster" ,
            bus=f"{node} battery cluster",
            carrier=n.stores.at[link_name, "carrier"],
            e_nom_extendable=True,
            capital_cost=n.stores.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.stores.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            e_initial_per_period=n.stores.at[link_name, "e_initial_per_period"],
            e_cyclic=n.stores.at[link_name, "e_cyclic"],
            e_cyclic_per_period=n.stores.at[link_name, "e_cyclic_per_period"],
            overwrite=True,
            )
    return n

n = add_cluster_storages(n, nodes_with_clusters, cluster_cost_reduction)





In [41]:
n.links["reversed"] = n.links["reversed"].fillna(False).astype(bool)


**Printing to Check**

In [42]:
n.links.loc[n.links["bus1"]=='EU methanol']

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
IT0 0 solid biomass biomass-to-methanol,IT0 0 solid biomass,EU methanol,,biomass-to-methanol,0.6100,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
IT0 1 solid biomass biomass-to-methanol,IT0 1 solid biomass,EU methanol,,biomass-to-methanol,0.6100,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
IT0 2 solid biomass biomass-to-methanol,IT0 2 solid biomass,EU methanol,,biomass-to-methanol,0.6100,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
IT1 0 solid biomass biomass-to-methanol,IT1 0 solid biomass,EU methanol,,biomass-to-methanol,0.6100,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
IT0 0 methanolisation,IT0 0 H2,EU methanol,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
IT0 1 methanolisation,IT0 1 H2,EU methanol,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
IT0 2 methanolisation,IT0 2 H2,EU methanol,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
IT1 0 methanolisation,IT1 0 H2,EU methanol,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
IT0 1 methanolisation cluster,IT0 1 H2 cluster,EU methanol,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


In [43]:
n.links.loc[n.links.index.str.contains("Electrolysis")]

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
IT0 0 H2 Electrolysis,IT0 0,IT0 0 H2,,H2 Electrolysis,0.6217,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
IT0 1 H2 Electrolysis,IT0 1,IT0 1 H2,,H2 Electrolysis,0.6217,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
IT0 2 H2 Electrolysis,IT0 2,IT0 2 H2,,H2 Electrolysis,0.6217,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
IT1 0 H2 Electrolysis,IT1 0,IT1 0 H2,,H2 Electrolysis,0.6217,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
IT0 1 H2 Electrolysis cluster,IT0 1 cluster,IT0 1 H2 cluster,,H2 Electrolysis,0.6217,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


In [44]:
n.links.loc[n.links["carrier"].str.contains('H2 Electrolysis')]

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
IT0 0 H2 Electrolysis,IT0 0,IT0 0 H2,,H2 Electrolysis,0.6217,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
IT0 1 H2 Electrolysis,IT0 1,IT0 1 H2,,H2 Electrolysis,0.6217,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
IT0 2 H2 Electrolysis,IT0 2,IT0 2 H2,,H2 Electrolysis,0.6217,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
IT1 0 H2 Electrolysis,IT1 0,IT1 0 H2,,H2 Electrolysis,0.6217,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
IT0 1 H2 Electrolysis cluster,IT0 1 cluster,IT0 1 H2 cluster,,H2 Electrolysis,0.6217,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


In [45]:
n.buses.loc[n.buses.index.str.contains("cluster")]

,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network,country,substation_lv,substation_off
Bus,,,,,,,,,,,,,,,,
IT0 1 cluster,380.0,,15.089899,40.03572,AC,MWh_el,IT0 1,1.0,0.0,inf,PQ,,,IT,1.0,1.0
IT0 1 H2 cluster,1.0,,15.089899,40.03572,H2,MWh_LHV,IT0 1,1.0,0.0,inf,PQ,,,IT,NaN,NaN
IT0 1 battery cluster,1.0,,15.089899,40.03572,battery,MWh_el,IT0 1,1.0,0.0,inf,PQ,,,IT,NaN,NaN


In [46]:
n.stores.loc[n.stores.index.str.contains("cluster")]



,bus,type,carrier,e_nom,e_nom_mod,e_nom_extendable,e_nom_min,e_nom_max,e_min_pu,e_max_pu,...,marginal_cost,marginal_cost_quadratic,marginal_cost_storage,capital_cost,standing_loss,active,build_year,lifetime,e_nom_opt,location
Store,,,,,,,,,,,,,,,,,,,,,
IT0 1 H2 Store cluster,IT0 1 H2 cluster,,H2 Store,0.0,0.0,True,0.0,inf,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,True,0,inf,0.0,NaN
IT0 1 battery cluster,IT0 1 battery cluster,,battery,0.0,0.0,True,0.0,inf,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,True,0,inf,0.0,NaN


In [47]:
n.links.loc[n.links["carrier"]=='DC']

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
relation/8185711-500-DC,IT1 0,IT0 2,,DC,0.967388,True,0,inf,1000.0,0.0,...,0.0,relation/8185711,LINESTRING (8.306507186431089 40.8411086571579...,1.0,0.961843,,NaN,,False,556.670095
TYNDP2024_1157,IT0 0,IT0 2,,DC,0.974075,True,2030,inf,0.0,0.0,...,1.0,"{name:HG North Tyrrhenian Corridor,url:{name:T...","LINESTRING (9.409 45.553, 12.015 42.244)",NaN,0.000000,in_permitting,NaN,,False,260.622634
TYNDP2024_1166,IT0 2,IT0 1,,DC,0.968106,True,2036,inf,0.0,0.0,...,1.0,"{name:HG Adriatic Corridor,url:{name:TYNDP2024...","LINESTRING (11.661 44.855, 15.55 41.513)",NaN,0.330000,in_permitting,NaN,,False,524.800355
TYNDP2024_1168,IT0 1,IT0 2,,DC,0.968106,True,2035,inf,0.0,0.0,...,1.0,"{name:HG Ionian-Tyrrhenian Corridor,url:{name:...","LINESTRING (16.629 39.568, 12.779 41.43)",NaN,0.520000,in_permitting,NaN,,False,524.800355
relation/8185711-500-DC-reversed,IT0 2,IT1 0,,DC,0.967388,True,0,inf,1000.0,0.0,...,0.0,relation/8185711,LINESTRING (8.306507186431089 40.8411086571579...,1.0,0.961843,,NaN,,True,556.670095
TYNDP2024_1157-reversed,IT0 2,IT0 0,,DC,0.974075,True,2030,inf,0.0,0.0,...,1.0,"{name:HG North Tyrrhenian Corridor,url:{name:T...","LINESTRING (9.409 45.553, 12.015 42.244)",NaN,0.000000,in_permitting,NaN,,True,260.622634
TYNDP2024_1166-reversed,IT0 1,IT0 2,,DC,0.968106,True,2036,inf,0.0,0.0,...,1.0,"{name:HG Adriatic Corridor,url:{name:TYNDP2024...","LINESTRING (11.661 44.855, 15.55 41.513)",NaN,0.330000,in_permitting,NaN,,True,524.800355
TYNDP2024_1168-reversed,IT0 2,IT0 1,,DC,0.968106,True,2035,inf,0.0,0.0,...,1.0,"{name:HG Ionian-Tyrrhenian Corridor,url:{name:...","LINESTRING (16.629 39.568, 12.779 41.43)",NaN,0.520000,in_permitting,NaN,,True,524.800355


In [48]:
n.stores

,bus,type,carrier,e_nom,e_nom_mod,e_nom_extendable,e_nom_min,e_nom_max,e_min_pu,e_max_pu,...,marginal_cost,marginal_cost_quadratic,marginal_cost_storage,capital_cost,standing_loss,active,build_year,lifetime,e_nom_opt,location
Store,,,,,,,,,,,,,,,,,,,,,
co2 atmosphere,co2 atmosphere,,co2,0.000000e+00,0.0,True,0.000000e+00,inf,-1.0,1.0,...,0.0,0.0,0.0,0.000000,0.000000,True,0,inf,0.0,
IT0 0 co2 stored,IT0 0 co2 stored,,co2 stored,0.000000e+00,0.0,True,0.000000e+00,inf,0.0,1.0,...,0.0,0.0,0.0,247.607546,0.000000,True,0,inf,0.0,
IT0 1 co2 stored,IT0 1 co2 stored,,co2 stored,0.000000e+00,0.0,True,0.000000e+00,inf,0.0,1.0,...,0.0,0.0,0.0,247.607546,0.000000,True,0,inf,0.0,
IT0 2 co2 stored,IT0 2 co2 stored,,co2 stored,0.000000e+00,0.0,True,0.000000e+00,inf,0.0,1.0,...,0.0,0.0,0.0,247.607546,0.000000,True,0,inf,0.0,
IT1 0 co2 stored,IT1 0 co2 stored,,co2 stored,0.000000e+00,0.0,True,0.000000e+00,inf,0.0,1.0,...,0.0,0.0,0.0,247.607546,0.000000,True,0,inf,0.0,
IT0 0 co2 sequestered,IT0 0 co2 sequestered,,co2 sequestered,0.000000e+00,0.0,True,0.000000e+00,0.000000e+00,0.0,1.0,...,-0.1,0.0,0.0,30.000000,0.000000,True,0,50.0,0.0,
IT0 1 co2 sequestered,IT0 1 co2 sequestered,,co2 sequestered,0.000000e+00,0.0,True,0.000000e+00,1.990137e+07,0.0,1.0,...,-0.1,0.0,0.0,30.000000,0.000000,True,0,50.0,0.0,
IT0 2 co2 sequestered,IT0 2 co2 sequestered,,co2 sequestered,0.000000e+00,0.0,True,0.000000e+00,1.400831e+07,0.0,1.0,...,-0.1,0.0,0.0,30.000000,0.000000,True,0,50.0,0.0,
IT1 0 co2 sequestered,IT1 0 co2 sequestered,,co2 sequestered,0.000000e+00,0.0,True,0.000000e+00,0.000000e+00,0.0,1.0,...,-0.1,0.0,0.0,30.000000,0.000000,True,0,50.0,0.0,


In [49]:
n.links.loc[n.links["bus0"]=='EU methanol']

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
IT0 0 OCGT methanol,EU methanol,IT0 0,,OCGT methanol,0.41,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
IT0 1 OCGT methanol,EU methanol,IT0 1,,OCGT methanol,0.41,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
IT0 2 OCGT methanol,EU methanol,IT0 2,,OCGT methanol,0.41,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
IT1 0 OCGT methanol,EU methanol,IT1 0,,OCGT methanol,0.41,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
EU industry methanol,EU methanol,EU industry methanol,,industry methanol,1.00,True,0,inf,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
EU shipping methanol,EU methanol,EU shipping methanol,,shipping methanol,1.00,True,0,inf,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0


In [50]:
n.carriers

,co2_emissions,color,nice_name,max_growth,max_relative_growth
Carrier,,,,,
AC,0.0,#70af1d,AC,inf,0.0
DC,0.0,#8a1caf,DC,inf,0.0
solar,0.0,#f9d002,Solar,inf,0.0
onwind,0.0,#235ebc,Onshore Wind,inf,0.0
offwind-float,0.0,#b5e2fa,Offshore Wind (Floating),inf,0.0
...,...,...,...,...,...
H2 Store,0.0,#bf13a0,H2 Store,inf,0.0
urban decentral heat vent,0.0,#aa3344,urban decentral heat vent,inf,0.0
solar rooftop,0.0,#ffea80,solar rooftop,inf,0.0


In [51]:
n.global_constraints

,type,investment_period,carrier_attribute,sense,constant,mu
GlobalConstraint,,,,,,
lv_limit,transmission_volume_expansion_limit,NaN,"AC, DC",<=,1.278665e+07,0.0
biomass limit,operational_limit,NaN,solid biomass,<=,5.378019e+07,0.0
CO2Limit,co2_atmosphere,NaN,co2_emissions,<=,0.000000e+00,0.0


In [52]:
n.loads

,bus,carrier,type,p_set,q_set,sign,active
Load,,,,,,,
IT0 0,IT0 0 low voltage,electricity,,0.0,0.0,-1.0,True
IT0 1,IT0 1 low voltage,electricity,,0.0,0.0,-1.0,True
IT0 2,IT0 2 low voltage,electricity,,0.0,0.0,-1.0,True
IT1 0,IT1 0 low voltage,electricity,,0.0,0.0,-1.0,True
IT0 0 land transport EV,IT0 0 EV battery,land transport EV,,0.0,0.0,-1.0,True
...,...,...,...,...,...,...,...
IT0 1 urban decentral heat,IT0 1 urban decentral heat,urban decentral heat,,0.0,0.0,-1.0,True
IT0 2 rural heat,IT0 2 rural heat,rural heat,,0.0,0.0,-1.0,True
IT0 2 urban decentral heat,IT0 2 urban decentral heat,urban decentral heat,,0.0,0.0,-1.0,True


In [53]:
n.links

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
relation/8185711-500-DC,IT1 0,IT0 2,,DC,0.967388,True,0,inf,1000.0,0.0,...,0.0,relation/8185711,LINESTRING (8.306507186431089 40.8411086571579...,1.0,0.961843,,NaN,,False,556.670095
TYNDP2024_1157,IT0 0,IT0 2,,DC,0.974075,True,2030,inf,0.0,0.0,...,1.0,"{name:HG North Tyrrhenian Corridor,url:{name:T...","LINESTRING (9.409 45.553, 12.015 42.244)",NaN,0.000000,in_permitting,NaN,,False,260.622634
TYNDP2024_1166,IT0 2,IT0 1,,DC,0.968106,True,2036,inf,0.0,0.0,...,1.0,"{name:HG Adriatic Corridor,url:{name:TYNDP2024...","LINESTRING (11.661 44.855, 15.55 41.513)",NaN,0.330000,in_permitting,NaN,,False,524.800355
TYNDP2024_1168,IT0 1,IT0 2,,DC,0.968106,True,2035,inf,0.0,0.0,...,1.0,"{name:HG Ionian-Tyrrhenian Corridor,url:{name:...","LINESTRING (16.629 39.568, 12.779 41.43)",NaN,0.520000,in_permitting,NaN,,False,524.800355
IT0 0 co2 sequestered,IT0 0 co2 stored,IT0 0 co2 sequestered,,co2 sequestered,1.000000,True,0,inf,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
IT1 0 urban decentral water tanks discharger,IT1 0 urban decentral water tanks,IT1 0 urban decentral heat,,urban decentral water tanks discharger,1.000000,True,0,30.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.000000
IT0 1 H2 Electrolysis cluster,IT0 1 cluster,IT0 1 H2 cluster,,H2 Electrolysis,0.621700,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
IT0 1 methanolisation cluster,IT0 1 H2 cluster,EU methanol,,methanolisation,0.878700,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


**Exporting**

In [54]:
n.export_to_netcdf(fn)


INFO:pypsa.network.io:Exported network 'Unnamed Network'saved to 'resources/IT_test/networks/base_s_4__168hh_2050.nc contains: stores, links, generators, global_constraints, storage_units, buses, lines, carriers, loads


<xarray.Dataset> Size: 9MB
Dimensions:                               (snapshots: 8760,
                                           investment_periods: 0, stores_i: 54,
                                           stores_t_e_min_pu_i: 4,
                                           stores_t_e_max_pu_i: 8,
                                           links_i: 284,
                                           links_t_efficiency_i: 16,
                                           ...
                                           global_constraints_i: 3,
                                           storage_units_i: 8,
                                           storage_units_t_inflow_i: 4,
                                           buses_i: 111, lines_i: 3,
                                           carriers_i: 119, loads_i: 71,
                                           loads_t_p_set_i: 20)
Coordinates: (12/18)
  * snapshots                             (snapshots) int64 70kB 0 1 ... 8759
  * investment_periods                    (investment_periods) object 0B 
  * stores_i                              (stores_i) object 432B 'co2 atmosph...
  * stores_t_e_min_pu_i                   (stores_t_e_min_pu_i) object 32B 'I...
  * stores_t_e_max_pu_i                   (stores_t_e_max_pu_i) object 64B 'I...
  * links_i                               (links_i) object 2kB 'relation/8185...
    ...                                    ...
  * storage_units_t_inflow_i              (storage_units_t_inflow_i) object 32B ...
  * buses_i                               (buses_i) object 888B 'IT0 0' ... '...
  * lines_i                               (lines_i) object 24B '0' '1' '2'
  * carriers_i                            (carriers_i) object 952B 'AC' ... '...
  * loads_i                               (loads_i) object 568B 'IT0 0' ... '...
  * loads_t_p_set_i                       (loads_t_p_set_i) object 160B 'IT0 ...
Data variables: (12/124)
    snapshots_snapshot                    (snapshots) datetime64[ns] 70kB 201...
    snapshots_objective                   (snapshots) float64 70kB 1.0 ... 1.0
    snapshots_stores                      (snapshots) float64 70kB 1.0 ... 1.0
    snapshots_generators                  (snapshots) float64 70kB 1.0 ... 1.0
    investment_periods_objective          (investment_periods) float64 0B 
    investment_periods_years              (investment_periods) float64 0B 
    ...                                    ...
    carriers_color                        (carriers_i) object 952B '#70af1d' ...
    carriers_nice_name                    (carriers_i) object 952B 'AC' ... '...
    loads_bus                             (loads_i) object 568B 'IT0 0 low vo...
    loads_carrier                         (loads_i) object 568B 'electricity'...
    loads_p_set                           (loads_i) float64 568B 0.0 0.0 ... 0.0
    loads_t_p_set                         (snapshots, loads_t_p_set_i) float64 1MB ...
Attributes:
    network__multi_invest:  0
    network_name:           Unnamed Network
    network_pypsa_version:  0.35.2
    network_srid:           4326
    crs:                    {"_crs": "GEOGCRS[\"WGS 84\",ENSEMBLE[\"World Geo...
    meta:                   {"version": "v2025.07.0", "tutorial": false, "log...